In [ ]:
# =================================================================
# CELL 1: IB CONNECTION + DATA ENGINE
# =================================================================
import subprocess, sys
for p in ['ib_insync','pandas','numpy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])

from ib_insync import *
import pandas as pd, numpy as np, warnings, json
from datetime import datetime, timedelta
from IPython.display import display, HTML, clear_output
warnings.filterwarnings('ignore')
util.startLoop()

# Connect with clientId=0 (master client)
import os
IB_HOST = os.environ.get('IB_HOST', '127.0.0.1')
IB_PORT = int(os.environ.get('IB_PORT', '4002'))
IB_CLIENT = int(os.environ.get('IB_CLIENT', '0'))
print(f'Config: {IB_HOST}:{IB_PORT} client={IB_CLIENT}')

ib = IB()
try:
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT, timeout=10)
    print(f'Connected: {ib.isConnected()} | Account: {ib.managedAccounts()}')
except Exception as e:
    print(f'IB connection failed: {e}')
    print('Running in DEMO mode with simulated data')
print(f'Connected: {ib.isConnected()} | Account: {ib.managedAccounts()}')

# Get account data
account_summary = {}
portfolio_items = []
for av in ib.accountSummary():
    account_summary[av.tag] = av.value
portfolio_items = ib.portfolio()
print(f'Account fields: {len(account_summary)} | Positions: {len(portfolio_items)}')

# Get NQ contract
nq_dets = ib.reqContractDetails(Future(symbol='NQ', exchange='CME'))
nq_dets = [c for c in nq_dets if c.contract.tradingClass=='NQ']
nq_dets.sort(key=lambda c: c.contract.lastTradeDateOrContractMonth)
nq = nq_dets[0].contract
print(f'NQ: {nq.localSymbol} exp={nq.lastTradeDateOrContractMonth}')

# Try historical data
df_1m = pd.DataFrame()
DATA_SRC = 'NONE'
for wts in ['TRADES','MIDPOINT','BID']:
    bars = ib.reqHistoricalData(nq, endDateTime='', durationStr='2 D',
        barSizeSetting='1 min', whatToShow=wts, useRTH=False,
        formatDate=1, keepUpToDate=False, timeout=8)
    if bars:
        df_1m = util.df(bars); df_1m.set_index('date',inplace=True)
        DATA_SRC = f'IB_{wts}'; print(f'{wts}: {len(df_1m)} bars'); break
    print(f'{wts}: 0 bars')

if len(df_1m)==0:
    DATA_SRC='SIMULATED'
    print('Market data unavailable - generating simulated NQ bars')
    np.random.seed(42); n=2880
    dates=pd.date_range(end=datetime.now(),periods=n,freq='1min')
    p=21500.0; px=[p]
    for i in range(1,n): p+=np.random.randn()*5; px.append(p)
    px=np.array(px)
    df_1m=pd.DataFrame({'open':px+np.random.randn(n)*2,
        'high':px+abs(np.random.randn(n)*8),'low':px-abs(np.random.randn(n)*8),
        'close':px,'volume':np.random.randint(100,5000,n).astype(float)},index=dates)

print(f'\nREADY | Source: {DATA_SRC} | Bars: {len(df_1m)}')

In [ ]:
# =================================================================
# CELL 2: INDICATORS + SIGNAL ENGINE
# =================================================================

def calc_rsi(s, period=14):
    d = s.diff()
    g = d.where(d>0,0).ewm(alpha=1/period,adjust=False).mean()
    l = (-d.where(d<0,0)).ewm(alpha=1/period,adjust=False).mean()
    return 100 - 100/(1+g/l.replace(0,np.nan))

def calc_stoch(df, k=14, d=3):
    lo = df['low'].rolling(k).min()
    hi = df['high'].rolling(k).max()
    sk = 100*(df['close']-lo)/(hi-lo)
    sd = sk.rolling(d).mean()
    return sk, sd

def calc_macd(s, fast=12, slow=26, sig=9):
    ef = s.ewm(span=fast,adjust=False).mean()
    es = s.ewm(span=slow,adjust=False).mean()
    ml = ef-es; sl = ml.ewm(span=sig,adjust=False).mean()
    return ml, sl, ml-sl

def add_indicators(df):
    d = df.copy()
    d['rsi'] = calc_rsi(d['close'])
    d['stoch_k'], d['stoch_d'] = calc_stoch(d)
    d['macd'], d['macd_sig'], d['macd_hist'] = calc_macd(d['close'])
    return d.dropna()

def resample(df, rule):
    return df.resample(rule).agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna()

# Build multi-timeframe data
df_1m_ind  = add_indicators(df_1m)
df_30m_ind = add_indicators(resample(df_1m, '30min'))
df_1h_ind  = add_indicators(resample(df_1m, '60min'))

print(f'1m: {len(df_1m_ind)} | 30m: {len(df_30m_ind)} | 1h: {len(df_1h_ind)}')

# --- SIGNAL ENGINE ---
def get_signal():
    if len(df_1h_ind)<2 or len(df_30m_ind)<2 or len(df_1m_ind)<2:
        return 'NO DATA', {}
    s1  = df_1m_ind.iloc[-1]
    s30 = df_30m_ind.iloc[-1]
    s60 = df_1h_ind.iloc[-1]
    info = {'1m_rsi':s1.rsi,'30m_rsi':s30.rsi,'1h_rsi':s60.rsi,
            '1m_stk':s1.stoch_k,'1m_std':s1.stoch_d,
            '30m_macd_h':s30.macd_hist,'1h_macd_h':s60.macd_hist,
            '1m_macd_h':s1.macd_hist, 'price':s1.close}
    
    # BULLISH: sell PUT credit spread (collect premium below market)
    bull = (s60.rsi>50 and s30.rsi>50 and
            s60.macd>s60.macd_sig and s30.macd>s30.macd_sig and
            s1.macd_hist>0 and s1.stoch_k>s1.stoch_d)
    
    # BEARISH: sell CALL credit spread (collect premium above market)
    bear = (s60.rsi<50 and s30.rsi<50 and
            s60.macd<s60.macd_sig and s30.macd<s30.macd_sig and
            s1.macd_hist<0 and s1.stoch_k<s1.stoch_d)
    
    if bull and not bear: return 'SELL PUT SPREAD (30d/20d)', info
    if bear and not bull: return 'SELL CALL SPREAD (30d/20d)', info
    return 'NEUTRAL - NO TRADE', info

signal, details = get_signal()
print(f'Signal: {signal}')
for k,v in details.items(): print(f'  {k}: {v:.2f}')

In [ ]:
# =================================================================
# CELL 3b: FIXED DASHBOARD (no backslashes in f-strings)
# =================================================================
UP = chr(9650); DOWN = chr(9660); DOT = chr(9679); CHECK = chr(9989); CROSS = chr(10060); DELTA = chr(916); BULL = chr(8226)

def render_dashboard():
    signal, info = get_signal()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    if 'PUT' in signal: sig_color,sig_bg,sig_icon='#00c853','#1b3a1b',UP
    elif 'CALL' in signal: sig_color,sig_bg,sig_icon='#ff1744','#3a1b1b',DOWN
    else: sig_color,sig_bg,sig_icon='#ffd740','#3a3520',DOT
    
    nlv=account_summary.get('NetLiquidation','N/A')
    cash=account_summary.get('TotalCashValue','N/A')
    bp=account_summary.get('BuyingPower','N/A')
    unrpnl=account_summary.get('UnrealizedPnL','0')
    rpnl=account_summary.get('RealizedPnL','0')
    maint=account_summary.get('MaintMarginReq','N/A')
    conn_icon = CHECK if ib.isConnected() else CROSS
    conn_txt = 'Connected' if ib.isConnected() else 'Disconnected'
    
    def fmt(v):
        try: return f"${float(v):,.2f}"
        except: return str(v)
    def rsi_c(v):
        if v>70: return '#ff1744'
        if v<30: return '#00c853'
        return '#ffd740'
    def bar(val,mn,mx,color):
        pct=max(0,min(100,(val-mn)/(mx-mn)*100))
        return f'<div style="background:#333;border-radius:3px;height:8px;width:100%"><div style="background:{color};height:8px;border-radius:3px;width:{pct}%"></div></div>'
    
    def tf_block(name,df):
        r=df.iloc[-1]
        rc=rsi_c(r.rsi)
        mh_c='#00c853' if r.macd_hist>0 else '#ff1744'
        ms_c='#00c853' if r.macd>r.macd_sig else '#ff1744'
        ms_t=UP+' ABOVE' if r.macd>r.macd_sig else DOWN+' BELOW'
        return f'''<div style="background:#1e1e1e;border-radius:8px;padding:12px;margin:4px 0">
<div style="font-weight:700;color:#90caf9;margin-bottom:6px">{name}</div>
<table style="width:100%;color:#e0e0e0;font-size:13px">
<tr><td>RSI</td><td style="color:{rc}">{r.rsi:.1f}</td><td style="width:60%">{bar(r.rsi,0,100,rc)}</td></tr>
<tr><td>Stoch K/D</td><td>{r.stoch_k:.1f}/{r.stoch_d:.1f}</td><td>{bar(r.stoch_k,0,100,'#ce93d8')}</td></tr>
<tr><td>MACD Hist</td><td style="color:{mh_c}">{r.macd_hist:.2f}</td><td>{bar(r.macd_hist+20,0,40,mh_c)}</td></tr>
<tr><td>MACD/Sig</td><td>{r.macd:.2f}/{r.macd_sig:.2f}</td><td style="color:{ms_c}">{ms_t}</td></tr>
</table></div>'''
    
    if portfolio_items:
        pos_rows=''
        for p in portfolio_items:
            pc='#00c853' if p.unrealizedPNL>=0 else '#ff1744'
            pos_rows+=f'<tr><td>{p.contract.symbol}</td><td>{p.position}</td><td>{fmt(p.marketValue)}</td><td style="color:{pc}">{fmt(p.unrealizedPNL)}</td></tr>'
        pos_html=f'<div style="background:#1e1e1e;border-radius:8px;padding:12px;margin-top:8px"><div style="font-weight:700;color:#90caf9;margin-bottom:6px">Open Positions</div><table style="width:100%;color:#e0e0e0;font-size:13px"><tr style="color:#757575"><th align=left>Symbol</th><th>Qty</th><th>Value</th><th>PnL</th></tr>{pos_rows}</table></div>'
    else:
        pos_html='<div style="background:#1e1e1e;border-radius:8px;padding:12px;margin-top:8px;color:#757575;text-align:center">No open positions</div>'
    
    if 'PUT' in signal:
        spread_html=f'<div style="background:#1b3a1b;border:1px solid #00c853;border-radius:8px;padding:12px;margin-top:8px"><div style="color:#00c853;font-weight:700">{UP} SELL PUT CREDIT SPREAD on NQ</div><div style="color:#a5d6a7;font-size:13px;margin-top:4px">Short 30{DELTA} Put {BULL} Long 20{DELTA} Put {BULL} Bullish bias on all TFs</div></div>'
    elif 'CALL' in signal:
        spread_html=f'<div style="background:#3a1b1b;border:1px solid #ff1744;border-radius:8px;padding:12px;margin-top:8px"><div style="color:#ff1744;font-weight:700">{DOWN} SELL CALL CREDIT SPREAD on NQ</div><div style="color:#ef9a9a;font-size:13px;margin-top:4px">Short 30{DELTA} Call {BULL} Long 20{DELTA} Call {BULL} Bearish bias on all TFs</div></div>'
    else:
        spread_html=f'<div style="background:#3a3520;border:1px solid #ffd740;border-radius:8px;padding:12px;margin-top:8px"><div style="color:#ffd740;font-weight:700">{DOT} NO TRADE - Signals Not Aligned</div><div style="color:#fff9c4;font-size:13px;margin-top:4px">Waiting for RSI + Stochastic + MACD to align across 1h, 30m, 1m timeframes</div></div>'
    
    unrpnl_c='#00c853' if float(unrpnl or 0)>=0 else '#ff1744'
    rpnl_c='#00c853' if float(rpnl or 0)>=0 else '#ff1744'
    price=info.get('price',0)
    
    html=f'''
<div style="font-family:'Segoe UI',system-ui,sans-serif;background:#121212;color:#fff;padding:20px;border-radius:12px;max-width:920px">
<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:16px">
<div><h2 style="margin:0;color:#90caf9">NQ Credit Spread Signal Dashboard</h2>
<div style="color:#757575;font-size:12px">{now} | Data: {DATA_SRC} | IB: {conn_icon} {conn_txt}</div></div>
<div style="background:{sig_bg};border:2px solid {sig_color};border-radius:8px;padding:8px 16px;text-align:center">
<div style="font-size:24px">{sig_icon}</div>
<div style="color:{sig_color};font-weight:700;font-size:13px">{signal}</div></div></div>

<div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:8px;margin-bottom:12px">
<div style="background:#1e1e1e;border-radius:8px;padding:12px;text-align:center"><div style="color:#757575;font-size:11px">Net Liquidation</div><div style="color:#fff;font-size:18px;font-weight:700">{fmt(nlv)}</div></div>
<div style="background:#1e1e1e;border-radius:8px;padding:12px;text-align:center"><div style="color:#757575;font-size:11px">Cash</div><div style="color:#fff;font-size:18px;font-weight:700">{fmt(cash)}</div></div>
<div style="background:#1e1e1e;border-radius:8px;padding:12px;text-align:center"><div style="color:#757575;font-size:11px">Buying Power</div><div style="color:#fff;font-size:18px;font-weight:700">{fmt(bp)}</div></div>
<div style="background:#1e1e1e;border-radius:8px;padding:12px;text-align:center"><div style="color:#757575;font-size:11px">Margin Req</div><div style="color:#fff;font-size:18px;font-weight:700">{fmt(maint)}</div></div></div>

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-bottom:8px">
<div style="background:#1e1e1e;border-radius:8px;padding:12px;text-align:center"><div style="color:#757575;font-size:11px">Unrealized PnL</div><div style="color:{unrpnl_c};font-size:18px;font-weight:700">{fmt(unrpnl)}</div></div>
<div style="background:#1e1e1e;border-radius:8px;padding:12px;text-align:center"><div style="color:#757575;font-size:11px">Realized PnL</div><div style="color:{rpnl_c};font-size:18px;font-weight:700">{fmt(rpnl)}</div></div></div>

<div style="background:#1e1e1e;border-radius:8px;padding:8px 12px;margin-bottom:8px;text-align:center">
<span style="color:#90caf9;font-weight:700;font-size:28px">{nq.localSymbol}: {price:.2f}</span></div>

{spread_html}

<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;margin-top:8px">
<div>{tf_block('1 HOUR',df_1h_ind)}</div>
<div>{tf_block('30 MIN',df_30m_ind)}</div>
<div>{tf_block('1 MIN',df_1m_ind)}</div></div>

{pos_html}

<div style="color:#757575;font-size:11px;text-align:center;margin-top:12px">
Strategy: Sell 30{DELTA} short / 20{DELTA} long credit spreads when RSI+Stoch+MACD align across 1h, 30m, 1m</div>
</div>'''
    
    clear_output(wait=True)
    display(HTML(html))

render_dashboard()

In [ ]:
# =================================================================
# CELL 5: AUTO-REFRESH LOOP (run to keep dashboard updating)
# Press the Stop button or Kernel > Interrupt to stop
# =================================================================
import time

REFRESH_SECONDS = 30  # Update every 30 seconds

try:
    while True:
        # Re-fetch account data
        try:
            account_summary.clear()
            for av in ib.accountSummary():
                account_summary[av.tag] = av.value
            portfolio_items = ib.portfolio()
        except: pass
        
        # Re-compute indicators (if live data streaming, df_1m updates automatically)
        df_1m_ind  = add_indicators(df_1m)
        df_30m_ind = add_indicators(resample(df_1m, '30min'))
        df_1h_ind  = add_indicators(resample(df_1m, '60min'))
        
        render_dashboard()
        ib.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print('Dashboard stopped.')
    

In [ ]:
# =====================================================================
# CELL 6: COMPREHENSIVE RELIABILITY TEST
# =====================================================================
import traceback

results = []
def test(name, func):
    try:
        ok, msg = func()
        results.append((name, 'PASS' if ok else 'FAIL', msg))
    except Exception as e:
        results.append((name, 'ERROR', str(e)))

# 1. IB Connection
test('IB Connection', lambda: (ib.isConnected(), f'Connected={ib.isConnected()}'))

# 2. Account Summary
test('Account Summary', lambda: (len(account_summary) > 0, f'{len(account_summary)} fields loaded'))

# 3. Key Account Fields
for field in ['NetLiquidation','TotalCashValue','BuyingPower']:
    test(f'Acct Field: {field}', lambda f=field: (f in account_summary, account_summary.get(f, 'MISSING')))

# 4. Portfolio Items
test('Portfolio Access', lambda: (isinstance(portfolio_items, list), f'{len(portfolio_items)} positions'))

# 5. NQ Contract
test('NQ Contract', lambda: (nq is not None and nq.symbol == 'NQ', f'{nq.localSymbol}'))

# 6. Historical Data
test('Historical Data', lambda: (len(df_1m) > 0, f'{len(df_1m)} bars loaded'))

# 7. Indicators on 1m
test('RSI Computed', lambda: ('rsi' in df_1m_ind.columns, f'RSI last={df_1m_ind["rsi"].iloc[-1]:.2f}'))
test('Stoch Computed', lambda: ('stoch_k' in df_1m_ind.columns and 'stoch_d' in df_1m_ind.columns, f'K={df_1m_ind["stoch_k"].iloc[-1]:.2f} D={df_1m_ind["stoch_d"].iloc[-1]:.2f}'))
test('MACD Computed', lambda: ('macd' in df_1m_ind.columns and 'macd_sig' in df_1m_ind.columns, f'MACD={df_1m_ind["macd"].iloc[-1]:.2f}'))

# 8. Multi-TF resampling
test('30min Resample', lambda: (len(df_30m_ind) > 0, f'{len(df_30m_ind)} bars'))
test('1h Resample', lambda: (len(df_1h_ind) > 0, f'{len(df_1h_ind)} bars'))

# 9. Signal Engine
try:
    sig = get_signal()
    test('Signal Engine', lambda: (sig is not None, f'action={sig["action"]}'))
except:
    test('Signal Engine', lambda: (False, 'generate_signal() failed'))

# 10. Dashboard Render
try:
    render_dashboard()
    test('Dashboard Render', lambda: (True, 'Rendered OK'))
except Exception as e:
    test('Dashboard Render', lambda: (False, str(e)))

# 11. RSI Range Check
rsi_val = df_1m_ind['rsi'].dropna()
test('RSI Range [0-100]', lambda: (rsi_val.min() >= 0 and rsi_val.max() <= 100, f'min={rsi_val.min():.1f} max={rsi_val.max():.1f}'))

# 12. Stoch Range Check
sk = df_1m_ind['stoch_k'].dropna()
test('Stoch Range [0-100]', lambda: (sk.min() >= 0 and sk.max() <= 100, f'min={sk.min():.1f} max={sk.max():.1f}'))

# Print Results
print('='*60)
print('  RELIABILITY TEST RESULTS')
print('='*60)
passed = sum(1 for _,s,_ in results if s=='PASS')
total = len(results)
for name, status, msg in results:
    icon = '\u2705' if status=='PASS' else '\u274c'
    print(f'{icon} {status:5s} | {name:25s} | {msg}')
print('='*60)
print(f'  {passed}/{total} tests passed')
if passed == total:
    print('  ALL TESTS PASSED - Dashboard is reliable!')
else:
    print(f'  {total-passed} test(s) need attention')
print('='*60)